# Structure

We have split this implementation into two scripts. kinematics_functions.py contains only general functions and mathematical models that can be used independently of what robot arm you are analyzing. kuka_arm_analysis.ipynb contains all variables spesific to the kuka arm problem, and the implementation of code that answers the provided questions using the functions from kinematics_functions.py. 

# PACKAGES

In [7]:
%matplotlib widget
import numpy as np
from kinematics_functions import *

# VARIABLES
In the cell below, all required variables for the kuka arm are defined, including the DH table derived in question 1.

In [8]:
# d values (not provided in the problem, so these are just estimates)
d_3 = 1 # Upper arm length
d_5 = 1 # Forearm length
d_7 = 1 # Hand length

h = 0.001 # Step size for numerical differentiation

# DH_table
def DH(q):
    return np.array([
        # d,    theta,  a,  alpha
        [0,     q[0],   0,  np.pi/2 ],
        [0,     q[1],   0, -np.pi/2 ],
        [d_3,   q[2],   0, -np.pi/2 ],
        [0,     q[3],   0,  np.pi/2 ],
        [d_5,   q[4],   0,  np.pi/2 ],
        [0,     q[5],   0, -np.pi/2 ],
        [d_7,   q[6],   0,  0       ],
    ])



# Question 4

## Part 1 - Verifying the Geometric Jacobian by comparing to direct kinematics


In [9]:
q = np.array([0, 0, 0, 0, 0, 0, 0], dtype=float) # Example joint angles

J = geometric_jacobian_from_DH_table(DH(q))
print("Geometric Jacobian at q = [0, 0, 0, 0, 0, 0, 0]:")
print(np.array2string(J, precision=3, suppress_small=True, floatmode='fixed'))


# ---- Numerical verification via direct kinematics ----
N = len(q)
T_0 = n_T_n_from_DH_table(DH(q), 0, N)
p_0, R_0 = T_0[:3, 3], T_0[:3, :3]

J_num = np.zeros((6, N))
for i in range(N):
    q_h = q.copy()
    q_h[i] += h
    T_h = n_T_n_from_DH_table(DH(q_h), 0, N)
    dR = T_h[:3, :3] @ R_0.T   # ≈ I + [omega]_x * h for small h

    J_num[:3, i] = (T_h[:3, 3] - p_0) / h
    J_num[3:, i] = [dR[2,1] - dR[1,2],
                    dR[0,2] - dR[2,0],
                    dR[1,0] - dR[0,1]]
    J_num[3:, i] /= 2*h

print("\nNumerical Jacobian at q = [0, 0, 0, 0, 0, 0, 0]:")
print(np.array2string(J_num, precision=3, suppress_small=True, floatmode='fixed'))
print(f"\nMax difference vs analytical: {np.max(np.abs(J - J_num)):.2e}")


Geometric Jacobian at q = [0, 0, 0, 0, 0, 0, 0]:
[[ 0.000 -3.000  0.000  2.000  0.000 -1.000  0.000]
 [ 0.000  0.000  0.000  0.000  0.000  0.000  0.000]
 [ 0.000  0.000  0.000  0.000  0.000  0.000  0.000]
 [ 0.000  0.000  0.000  0.000  0.000  0.000  0.000]
 [ 0.000 -1.000  0.000  1.000  0.000 -1.000  0.000]
 [ 1.000  0.000  1.000  0.000  1.000  0.000  1.000]]

Numerical Jacobian at q = [0, 0, 0, 0, 0, 0, 0]:
[[ 0.000 -3.000  0.000  2.000  0.000 -1.000  0.000]
 [-0.000 -0.000  0.000  0.000 -0.000 -0.000  0.000]
 [ 0.000 -0.001  0.000 -0.001  0.000 -0.000  0.000]
 [ 0.000  0.000  0.000  0.000  0.000  0.000  0.000]
 [ 0.000 -1.000  0.000  1.000  0.000 -1.000  0.000]
 [ 1.000  0.000  1.000  0.000  1.000  0.000  1.000]]

Max difference vs analytical: 1.50e-03


We see that there is only a small difference between the geometric jacobian we calculated and the one we found via numerical differentiation and direct kinematics, indicating strongly that our expression for the numerical jacobian is correct.

## Part 2 - Inspecting the Jacobian close to singularities

In [ ]:
# Code for inspecting Jacobian rank at singularities

# Question 5

## Part 1 - CLIK implementation




## Part 2 - Validation against task 3

In [ ]:
# ---- CLIK execution ----
# Desired end-effector pose (static target: no feedforward velocity).
p_d     = np.array([1.5, 0.5, 1.5])
R_d     = np.eye(3)
p_d_dot = np.zeros(3)
omega_d = np.zeros(3)

# Initial configuration (slightly bent so the convergence path is visible).
q_start     = np.array([0.0, 0.5, 0.0, -0.5, 0.0, 0.5, 0.0])
q_dot_start = np.zeros(7)

# Jacobian as a function of q (recomputed each iteration inside CLIK).
J_func = lambda q: geometric_jacobian_from_DH_table(DH(q))

q_final, q_dot_final, q_history = CLIK(
    p_d, R_d, omega_d, p_d_dot, q_start, q_dot_start,
    J=J_func, DH_func=DH,
)

# Achieved pose for comparison.
T_final = n_T_n_from_DH_table(DH(q_final), 0, len(q_final))
p_final = T_final[:3, 3]
R_final = T_final[:3, :3]

fmt = dict(precision=3, suppress_small=True, floatmode='fixed')

print("==== Desired ====")
print("Position:", np.array2string(p_d, **fmt))
print("Rotation:")
print(np.array2string(R_d, **fmt))

print(f"\n==== Achieved by CLIK after {len(q_history) - 1} iterations ====")
print("Position:", np.array2string(p_final, **fmt))
print("Rotation:")
print(np.array2string(R_final, **fmt))

print("\n==== Joint configuration found by CLIK (rad) ====")
print(np.array2string(q_final, **fmt))

plot_CLIK_convergence(q_history, DH)
